# CS03e — Passive execution of the CS03c perp flow signal

Protocol: `discussions/Candle_Streaks/003e_claude_passive_fill_preregistration.md`.

CS03c TEST signals (BTC, ETH; 2026-07-01 → 2026-09-30) are executed with a limit order $k$ bp behind the last trade, filled only on a strict trade-through within the forecast bar, closed at the first trade at or after $t+H$. Random-side passive orders on every origin measure adverse selection without information.

Inputs: `<BINANCE_UM_DIR>/<ASSET>USDT_um_1m.parquet` (signal), `<BINANCE_UM_DIR>/aggTrades/<ASSET>USDT/*.parquet` (execution), spot 5m (only to reproduce the CS03c day split).

In [1]:
import os,sys,json,subprocess,importlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Execute notebook inside the repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import candle_streak_trade_flow as tf
from data_engineering.modeling import candle_streak_flow_attribution as ab
from data_engineering.modeling import candle_streak_perp_target as pt
from data_engineering.modeling import candle_streak_passive_fill as pf
for m in (tf,ab,pt,pf):importlib.reload(m)
assert pf.VERSION=='CS03e-0.1.0', pf.VERSION
cfg=pf.Specification();ptcfg=pt.Specification();abcfg=ab.Specification()
SYNTHETIC=os.getenv('CS03E_SYNTHETIC','0')=='1'
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='cs03e-');OUT=Path(tmp.name)/'03e'
else:
    from data_engineering.modeling import markov_lift_distributions as vd14
    OUT=ROOT/'notebooks'/'Candle_Streaks'/'03e'
OUT.mkdir(parents=True,exist_ok=True)
print('CS03e:',pf.VERSION,'CS03c:',pt.VERSION,'synthetic:',SYNTHETIC,'output:',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
checks=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_candle_streak_passive_fill.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(checks.stdout[-2000:]);print(checks.stderr[-1000:])
assert checks.returncode==0, 'CS03e tests failed; stop before historical evaluation'

CS03e: CS03e-0.1.0 CS03c: CS03c-0.1.0 synthetic: False output: D:\Projects\deribit_market_data\notebooks\Candle_Streaks\03e
..........                                                               [100%]
10 passed in 1.99s




## 1. Reproduce the CS03c signal (TRAIN fit, TEST origins inside the execution window)

In [2]:
W0=pd.Timestamp(cfg.window_start,tz='UTC');W1=pd.Timestamp(cfg.window_end,tz='UTC')+pd.Timedelta(days=1)
window_dates=pd.date_range(W0,W1-pd.Timedelta(days=1),freq='D')
ORDERS={};info=[]
rng=np.random.default_rng(cfg.seed)
if SYNTHETIC:
    TRADE_ROOT=Path(tmp.name)/'aggTrades'
    window_dates=window_dates[:6]
    for asset in cfg.assets:
        sym=f'{asset}USDT';(TRADE_ROOT/sym).mkdir(parents=True,exist_ok=True)
        for i,d in enumerate(pd.date_range(W0-pd.Timedelta(days=1),periods=8,freq='D')):
            t,p=pf.synthetic_trades(n_min=1440,seed=100*i+len(asset),per_min=40,start=str(d.date()))
            pd.DataFrame({'time_ms':t,'price':p,'qty':1.0,'is_buyer_maker':False}).to_parquet(pf.day_path(TRADE_ROOT,sym,d),index=False)
        o=pd.date_range(W0+pd.Timedelta(minutes=5),window_dates[-1]+pd.Timedelta(days=1),freq='5min')
        allo=pd.DataFrame({'state_time':o});allo['date']=allo.state_time.dt.floor('D')
        sig=allo.sample(frac=.1,random_state=1).sort_values('state_time')
        sig['side']=np.where(rng.random(len(sig))<.5,1.,-1.)
        allo['side']=np.where(rng.random(len(allo))<.5,1.,-1.)
        ORDERS[asset]={'signal':sig,'random':allo}
else:
    root=Path(os.environ['DERIBIT_DATA_ROOT']);hist=root/'history'/'price_5m'
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else root.parent/'Binance'
    TRADE_ROOT=BIN/'aggTrades'
    print('klines:',BIN,'aggTrades:',TRADE_ROOT)
    for asset in cfg.assets:
        pr,ff=pt.read_perp(BIN/f'{asset}USDT_um_1m.parquet',end=ptcfg.end)
        prices,_=vd14.load_returns(hist,(asset,),end=pd.Timestamp(ptcfg.end),min_days=365)
        sp=prices.loc[prices.underlying.eq(asset),['state_time','return_5m']].copy()
        px,sx=pt.build_panel(pr,ff),pt.build_panel(sp,ff)
        dates=np.array(sorted(set(tf.full_days(px))&set(tf.full_days(sx))))
        x,split=tf.split_panel(px,dates,abcfg.train_fraction,abcfg.valid_fraction)
        train=x[x.period.eq('train')]
        fit=ab.fit_ablation(train,C=abcfg.logistic_C,max_iter=abcfg.max_iter,tol=abcfg.tol)
        thr=pt.edge_threshold(train,fit,'price_vol_signed',ptcfg.edge_quantile)
        ev=x[x.period.eq('test')&x.scorable].copy()
        ev['edge']=pt.edge(ab.predict_ablation(ev,fit)['price_vol_signed'])
        ev=ev[(ev.state_time>=W0)&(ev.state_time<W1)]
        ev['date']=ev.state_time.dt.floor('D')
        sig=ev[ev.edge.abs()>=thr].copy();sig['side']=np.sign(sig.edge)
        allo=ev[['state_time','date']].copy();allo['side']=np.where(rng.random(len(allo))<.5,1.,-1.)
        ORDERS[asset]={'signal':sig[['state_time','date','side','edge']],'random':allo}
        info.append({'asset':asset,'split':split,'threshold_q90':thr,'window_origins':len(ev),'signal_orders':len(sig),
                     'long_share':float((sig.side>0).mean())})
        del pr,ff,prices,sp,px,sx,x,train,ev
display(pd.DataFrame(info))

klines: D:\MarketData\Binance aggTrades: D:\MarketData\Binance\aggTrades


,asset,split,threshold_q90,window_origins,signal_orders,long_share
0,BTC,"{'train_before': '2025-12-21 00:00:00+00:00', ...",0.068076,26496,3106,0.436252
1,ETH,"{'train_before': '2025-12-21 00:00:00+00:00', ...",0.083712,26496,3061,0.499837


## 2. Simulate fills

In [3]:
rows=[];missing=[]
for asset,arms in ORDERS.items():
    sym=f'{asset}USDT'
    for d in window_dates:
        try:t,p=pf.load_window(TRADE_ROOT,sym,d)
        except FileNotFoundError as e:
            missing.append(str(e));continue
        for arm,o in arms.items():
            od=o[o.date==d]
            if od.empty:continue
            oms=pf.to_ms(od.state_time)
            for k in cfg.offsets_bp:
                for H in cfg.hold_min:
                    r=pf.simulate(oms,od.side.to_numpy(float),t,p,k_bp=k,fill_min=cfg.fill_window_min,hold_min=H,markout_s=cfg.markout_s)
                    rows.append(r.assign(asset=asset,arm=arm,k_bp=k,hold_min=H,date=d))
        del t,p
    print(asset,'done')
sim=pd.concat(rows,ignore_index=True)
print('orders simulated:',len(sim),'missing trade days:',len(missing))
if missing:print(missing[:5])
assert not SYNTHETIC or len(sim)>0

BTC done
ETH done
orders simulated: 473272 missing trade days: 0


In [4]:
def all_days(dt,days):
    return dt.set_index('date').reindex(days).fillna({'orders':0,'trades':0,'gross_bp':0.,'net_bp':0.}).reset_index().rename(columns={'index':'date'})
have=sorted(sim.date.unique())
nb=len(cfg.offsets_bp)
import warnings;warnings.filterwarnings('ignore',message='All-NaN slice')
summ=[];daily=[]
for (asset,arm,k,H),g in sim.groupby(['asset','arm','k_bp','hold_min']):
    for cname in pf.COSTS:
        c=pf.cost_bp(cname,cfg)
        dt=all_days(pf.day_table(g,c,passive=True),have)
        s=pf.summarize(dt,cfg.bootstrap_reps,cfg.seed);sb=pf.summarize(dt,cfg.bootstrap_reps,cfg.seed,alpha=.05/nb)
        summ.append({'asset':asset,'arm':arm,'kind':'passive','k_bp':k,'hold_min':H,'cost':cname,'cost_round_trip_bp':c,**s,
                     'net_bp_per_day_lo_bonf':sb['net_bp_per_day_lo']})
        if arm=='signal' and cname=='maker_in_taker_out':daily.append(dt.assign(asset=asset,k_bp=k,hold_min=H))
    if k==cfg.offsets_bp[0]:
        for cname,c in (('taker_both',2*cfg.taker_cost_bp),('zero',0.)):
            dt=all_days(pf.day_table(g,c,passive=False),have)
            summ.append({'asset':asset,'arm':arm,'kind':'taker','k_bp':np.nan,'hold_min':H,'cost':cname,'cost_round_trip_bp':c,
                         **pf.summarize(dt,cfg.bootstrap_reps,cfg.seed)})
summary=pd.DataFrame(summ);daily=pd.concat(daily,ignore_index=True)
cols=['asset','arm','kind','k_bp','hold_min','cost','orders','trades','fill_rate','gross_bp_per_trade','net_bp_per_trade','net_bp_per_day','net_bp_per_day_lo','net_bp_per_day_hi']
print('H = 5, signal vs random, zero cost (gross; adverse selection shows as negative gross for random passive):')
display(summary[(summary.hold_min==5)&(summary.cost=='zero')][cols].round(3))
print('H = 5, gate cost (maker in 2 bp + taker out 6 bp) and taker benchmark (12 bp):')
display(summary[(summary.hold_min==5)&summary.cost.isin(['maker_in_taker_out','maker_both','taker_both'])&(summary.arm=='signal')][cols+['net_bp_per_day_lo_bonf']].round(3))

H = 5, signal vs random, zero cost (gross; adverse selection shows as negative gross for random passive):


,asset,arm,kind,k_bp,hold_min,cost,orders,trades,fill_rate,gross_bp_per_trade,net_bp_per_trade,net_bp_per_day,net_bp_per_day_lo,net_bp_per_day_hi
2,BTC,random,passive,0.0,5,zero,26496,24557,0.927,-0.606,-0.606,-161.680,-194.950,-128.060
4,BTC,random,taker,NaN,5,zero,26496,26494,1.000,-0.014,-0.014,-4.123,-38.064,30.178
12,BTC,random,passive,1.0,5,zero,26496,20623,0.778,-0.851,-0.851,-190.862,-226.620,-155.196
18,BTC,random,passive,2.0,5,zero,26496,17975,0.678,-0.753,-0.753,-147.169,-181.662,-112.076
24,BTC,random,passive,5.0,5,zero,26496,11710,0.442,-0.597,-0.597,-75.987,-109.810,-44.033
30,BTC,signal,passive,0.0,5,zero,3106,2799,0.901,-0.216,-0.216,-6.572,-20.054,5.191
32,BTC,signal,taker,NaN,5,zero,3106,3106,1.000,0.449,0.449,15.159,1.481,27.222
40,BTC,signal,passive,1.0,5,zero,3106,2293,0.738,-0.462,-0.462,-11.518,-24.270,-0.685
46,BTC,signal,passive,2.0,5,zero,3106,1928,0.621,-0.411,-0.411,-8.613,-21.198,2.747
52,BTC,signal,passive,5.0,5,zero,3106,1208,0.389,-0.228,-0.228,-2.991,-13.527,5.963


H = 5, gate cost (maker in 2 bp + taker out 6 bp) and taker benchmark (12 bp):


,asset,arm,kind,k_bp,hold_min,cost,orders,trades,fill_rate,gross_bp_per_trade,net_bp_per_trade,net_bp_per_day,net_bp_per_day_lo,net_bp_per_day_hi,net_bp_per_day_lo_bonf
28,BTC,signal,passive,0.0,5,maker_in_taker_out,3106,2799,0.901,-0.216,-8.216,-249.964,-272.265,-228.453,-278.945
29,BTC,signal,passive,0.0,5,maker_both,3106,2799,0.901,-0.216,-4.216,-128.268,-144.924,-112.495,-150.128
31,BTC,signal,taker,NaN,5,taker_both,3106,3106,1.000,0.449,-11.551,-389.971,-422.342,-359.274,NaN
38,BTC,signal,passive,1.0,5,maker_in_taker_out,3106,2293,0.738,-0.462,-8.462,-210.909,-230.090,-192.985,-234.139
39,BTC,signal,passive,1.0,5,maker_both,3106,2293,0.738,-0.462,-4.462,-111.214,-126.875,-97.606,-130.763
44,BTC,signal,passive,2.0,5,maker_in_taker_out,3106,1928,0.621,-0.411,-8.411,-176.265,-194.683,-159.384,-199.383
45,BTC,signal,passive,2.0,5,maker_both,3106,1928,0.621,-0.411,-4.411,-92.439,-107.648,-79.449,-111.282
50,BTC,signal,passive,5.0,5,maker_in_taker_out,3106,1208,0.389,-0.228,-8.228,-108.035,-123.497,-94.322,-127.479
51,BTC,signal,passive,5.0,5,maker_both,3106,1208,0.389,-0.228,-4.228,-55.513,-68.217,-44.649,-71.320
84,ETH,signal,passive,0.0,5,maker_in_taker_out,3061,2860,0.934,-0.514,-8.514,-264.666,-288.707,-239.558,-294.926


## 3. Markouts after a passive fill (bp, signed; negative = price moved against the filled order)

In [5]:
f=sim[sim.filled&(sim.hold_min==5)]
mk=f.groupby(['asset','arm','k_bp']).agg(n=('filled','size'),**{f'markout_{s}s_bp':(f'markout_{s}s',lambda v:1e4*np.nanmean(v)) for s in cfg.markout_s},
                                        **{f'se_{s}s_bp':(f'markout_{s}s',lambda v:1e4*np.nanstd(v)/np.sqrt(max(np.isfinite(v).sum(),1))) for s in cfg.markout_s}).reset_index()
display(mk.round(3))
fill_vs_taker=sim[(sim.hold_min==5)].groupby(['asset','arm','k_bp']).apply(lambda g:pd.Series({
    'taker_gross_bp_all':1e4*g.gross_taker.mean(),'taker_gross_bp_if_filled':1e4*g.loc[g.filled,'gross_taker'].mean(),
    'taker_gross_bp_if_unfilled':1e4*g.loc[~g.filled,'gross_taker'].mean()}),include_groups=False).reset_index()
print('Selection: the taker outcome of the same orders, split by whether the passive order would have filled')
display(fill_vs_taker.round(3))

,asset,arm,k_bp,n,markout_60s_bp,markout_300s_bp,se_60s_bp,se_300s_bp
0,BTC,random,0.0,24557,-0.650,-0.589,0.035,0.071
1,BTC,random,1.0,20623,-0.972,-0.853,0.040,0.081
2,BTC,random,2.0,17975,-0.947,-0.766,0.045,0.090
3,BTC,random,5.0,11710,-0.842,-0.505,0.064,0.125
4,BTC,signal,0.0,2799,-0.456,-0.246,0.110,0.219
5,BTC,signal,1.0,2293,-0.831,-0.442,0.132,0.262
6,BTC,signal,2.0,1928,-0.885,-0.278,0.154,0.301
7,BTC,signal,5.0,1208,-0.919,0.181,0.223,0.427
8,ETH,random,0.0,24867,-0.712,-0.684,0.048,0.094
9,ETH,random,1.0,22099,-0.928,-0.871,0.054,0.103


Selection: the taker outcome of the same orders, split by whether the passive order would have filled


,asset,arm,k_bp,taker_gross_bp_all,taker_gross_bp_if_filled,taker_gross_bp_if_unfilled
0,BTC,random,0.0,-0.014,-0.606,7.481
1,BTC,random,1.0,-0.014,-1.852,6.440
2,BTC,random,2.0,-0.014,-2.754,5.766
3,BTC,random,5.0,-0.014,-5.597,4.408
4,BTC,signal,0.0,0.449,-0.216,6.512
5,BTC,signal,1.0,0.449,-1.463,5.841
6,BTC,signal,2.0,0.449,-2.411,5.131
7,BTC,signal,5.0,0.449,-5.228,4.062
8,ETH,random,0.0,0.045,-0.673,11.013
9,ETH,random,1.0,0.045,-1.856,9.606


In [6]:
dec=pf.decision(summary,cfg,synthetic=SYNTHETIC)
print(json.dumps(dec,indent=2,default=str))

{
  "verdict": "NO_PASSIVE_EDGE",
  "passing_offsets_bp": [],
  "gates": [
    {
      "asset": "BTC",
      "k_bp": 0.0,
      "net_bp_per_day": -249.9637430038571,
      "lower_bonferroni": -278.94495065332876,
      "pass": false
    },
    {
      "asset": "ETH",
      "k_bp": 0.0,
      "net_bp_per_day": -264.6661364464083,
      "lower_bonferroni": -294.92623112887856,
      "pass": false
    },
    {
      "asset": "BTC",
      "k_bp": 1.0,
      "net_bp_per_day": -210.9094155896825,
      "lower_bonferroni": -234.139113325288,
      "pass": false
    },
    {
      "asset": "ETH",
      "k_bp": 1.0,
      "net_bp_per_day": -237.4217394832266,
      "lower_bonferroni": -266.06831443383095,
      "pass": false
    },
    {
      "asset": "BTC",
      "k_bp": 2.0,
      "net_bp_per_day": -176.26481716912437,
      "lower_bonferroni": -199.38330287358843,
      "pass": false
    },
    {
      "asset": "ETH",
      "k_bp": 2.0,
      "net_bp_per_day": -210.9490016682625,
      "low

In [7]:
aggregate={'03e_summary.csv':summary,'03e_markouts.csv':mk,'03e_fill_selection.csv':fill_vs_taker,'03e_daily_signal_gate_cost.csv':daily}
for name,frame in aggregate.items():frame.to_csv(OUT/name,index=False)
meta={'module':'CS03e','registration':pf.registration(),'cs03c':pt.registration(),'synthetic':SYNTHETIC,'signal_info':info,
      'missing_trade_days':missing,'days_evaluated':[str(d.date()) for d in have],'historical_test_reused':True}
(OUT/'03e_manifest.json').write_text(json.dumps(meta,indent=2,default=str),encoding='utf-8')
(OUT/'03e_decision.json').write_text(json.dumps(dec,indent=2,default=str),encoding='utf-8')
print('Saved',len(aggregate)+2,'outputs to',OUT)

Saved 6 outputs to D:\Projects\deribit_market_data\notebooks\Candle_Streaks\03e
